# 4 · Why do runs fail?

Triage for one agent: which steps fail, what the agent does next, and which runs leave the harness's intended process.

In [ ]:
import vega_agent as va

runs = va.Corpus(va.read("../data/copilot-model-b.jsonl"), "model-b")
runs

## Which steps fail?

`find` returns every match of a query, at any depth, as a table.

In [ ]:
failures = runs.find("tool!")
failures["match"].value_counts()

## What happens next?

The transitions of the failing runs show where the agent goes after each step. Failed steps are red.

In [ ]:
failing = runs.where("tool!")
failing.chart(mark="transitions")

## Which runs leave the harness?

The harness is itself an expression. Runs that do not conform to it are the ones to read first.

In [ ]:
harness = "model -> tool.read_range[] -> (model -> tool[])* -> model"
print(f"{runs.conformance(harness):.0%} of runs conform")

outside = va.Corpus([run for run in runs if not va.conforms(run, harness)], "outside the harness")
outside.chart(mark="variants", level="names")

## Hand them to process mining

The same runs export to XES for pm4py, ProM or Celonis.

In [ ]:
from pathlib import Path

Path("failing.xes").write_text(va.to_xes(failing))
len(failing), len(outside)